In [1]:
!pip install mysql.connector.python
!pip install pandas
!pip install numpy

In [9]:
import mysql.connector
import pandas as pd
import numpy as np
import json

myconnect = mysql.connector.connect(
    user = "root",
    password = "",
    port = "3307",
    host = "localhost"
)

mycursor = myconnect.cursor(buffered = True)



In [10]:
mycursor.execute("USE ubereats_restaurant")

In [11]:
mycursor.execute("""CREATE TABLE orders_data (
    order_id VARCHAR(100),
    restaurant_name VARCHAR(255),
    order_date DATE,
    order_value FLOAT,
    discount_used VARCHAR(10),
    payment_method VARCHAR(20))""")

In [12]:
#data = pd.read_json('C:\\Users\\welcome\\Desktop\\guvi\\uber_eats project\\cleaned_orderdata_version1.json')

with open('C:\\Users\\welcome\\Desktop\\guvi\\uber_eats project\\cleaned_orderdata_version1.json', 'r') as f:
    json_data = json.load(f)

data = pd.DataFrame(json_data)

In [15]:
print(data['order_date'].head())

0    1746489600000
1    1767830400000
2    1769040000000
3    1762732800000
4    1766361600000
Name: order_date, dtype: int64


In [16]:
#data['order_date'] = pd.to_datetime(data['order_date'])

#data['order_date'] = data['order_date'].dt.date

#data['order_date'] = data['order_date'].dt.strftime('%Y-%m-%d')

# Converting Unix timestamp in milliseconds to proper date format
# MySQL DATE column cannot store millisecond timestamps directly,
# so we convert them into Python date objects before insertion.

data['order_date'] = pd.to_datetime(
    data['order_date'],
    unit='ms'
).dt.date

data_to_insert = [tuple(x) for x in data[[
    'order_id',
    'restaurant_name',
    'order_date',
    'order_value',
    'discount_used',
    'payment_method'
]].values]

insert_query = """
INSERT INTO orders_data (
    order_id,
    restaurant_name,
    order_date,
    order_value,
    discount_used,
    payment_method
) VALUES (%s, %s, %s, %s, %s, %s)
"""

batch_size = 1000

for i in range(0, len(data_to_insert), batch_size):
    batch = data_to_insert[i:i+batch_size]
    mycursor.executemany(insert_query, batch)
    myconnect.commit()
    print(f"{i+len(batch)} rows uploaded...")

1000 rows uploaded...
2000 rows uploaded...
3000 rows uploaded...
4000 rows uploaded...
5000 rows uploaded...
6000 rows uploaded...
7000 rows uploaded...
8000 rows uploaded...
9000 rows uploaded...
10000 rows uploaded...
11000 rows uploaded...
12000 rows uploaded...
13000 rows uploaded...
14000 rows uploaded...
15000 rows uploaded...
16000 rows uploaded...
17000 rows uploaded...
18000 rows uploaded...
19000 rows uploaded...
20000 rows uploaded...
21000 rows uploaded...
22000 rows uploaded...
23000 rows uploaded...
24000 rows uploaded...
25000 rows uploaded...


In [21]:
data.columns

Index(['order_id', 'restaurant_name', 'order_date', 'order_value',
       'discount_used', 'payment_method'],
      dtype='object')

In [18]:
#dynamic question and answer:
#Q1 - Which restaurants are the top 10 revenue generators, and how much total income have they contributed?

from tabulate import tabulate

mycursor.execute("""SELECT restaurant_name,SUM(order_value) total_revenue
FROM orders_data
GROUP BY restaurant_name
ORDER BY total_revenue DESC
LIMIT 10""")

out = mycursor.fetchall()
print(tabulate(out, headers = [i[0] for i in mycursor.description], tablefmt='psql'))

+--------------------------------+-----------------+
| restaurant_name                |   total_revenue |
|--------------------------------+-----------------|
| kesar sweet shop and fast food |         25222.4 |
| khan saheb grills and rolls    |         23507.2 |
| bob's bar                      |         19518.1 |
| cake cafe                      |         19335.2 |
| biryani mane                   |         19249.5 |
| hungry lee                     |         19167.5 |
| andhra grills                  |         19153.7 |
| mighty paws                    |         18410.5 |
| kkr foodies                    |         18137.2 |
| delhi ke bawarchi              |         18131.7 |
+--------------------------------+-----------------+


In [19]:
#Q2 - Which restaurants have processed the highest volume of orders, and what is the total count for each?

mycursor.execute("""SELECT restaurant_name,COUNT(order_id) total_orders 
FROM orders_data
GROUP BY restaurant_name
ORDER BY total_orders DESC
LIMIT 10""")

out = mycursor.fetchall()
print(tabulate(out, headers = [i[0] for i in mycursor.description], tablefmt='psql'))

+------------------------------------+----------------+
| restaurant_name                    |   total_orders |
|------------------------------------+----------------|
| khan saheb grills and rolls        |             25 |
| kesar sweet shop and fast food     |             20 |
| cake cafe                          |             19 |
| andhra grills                      |             19 |
| kkr foodies                        |             19 |
| hungry lee                         |             18 |
| orbis restaurant                   |             18 |
| reddy's restaurant                 |             18 |
| shree muthahalli veg               |             18 |
| jw kitchen - jw marriott bengaluru |             17 |
+------------------------------------+----------------+


In [20]:
#Q3 - How does the average order value (AOV) vary across different top-performing restaurants?

mycursor.execute("""SELECT restaurant_name,ROUND(AVG(order_value),2) avg_order_value
FROM orders_data
GROUP BY restaurant_name
ORDER BY avg_order_value DESC
LIMIT 10""")

out = mycursor.fetchall()
print(tabulate(out, headers = [i[0] for i in mycursor.description], tablefmt='psql'))

+--------------------------------------+-------------------+
| restaurant_name                      |   avg_order_value |
|--------------------------------------+-------------------|
| chatori tongue                       |           1690.9  |
| 1441 pizzeria                        |           1625.12 |
| shagun sweets & foods                |           1614.57 |
| chow san                             |           1610.83 |
| nando's                              |           1593.39 |
| chaiwala                             |           1593.3  |
| moto store & cafãâãâãâãâãâãâãâãâãâãâãâãâãâãâãâãâ©                                      |           1589.89 |
| biting junction                      |           1572.31 |
| bikaner jn                           |           1554.2  |
| salt - indian restaurant bar & grill |           1550.46 |
+--------------------------------------+-------------------+


In [22]:
#Q4 - How does the use of discounts impact the average order value and the total number of orders placed?

mycursor.execute("""SELECT discount_used,ROUND(AVG(order_value),2) avg_order_value,COUNT(*) total_orders
FROM orders_data
GROUP BY discount_used""")

out = mycursor.fetchall()
print(tabulate(out, headers = [i[0] for i in mycursor.description], tablefmt='psql'))

+-----------------+-------------------+----------------+
|   discount_used |   avg_order_value |   total_orders |
|-----------------+-------------------+----------------|
|               0 |            822.49 |          12509 |
|               1 |           1149.91 |          12491 |
+-----------------+-------------------+----------------+


In [23]:
#What are the most preferred payment methods used by customers, and which one dominates the transactions?

mycursor.execute("""SELECT payment_method, COUNT(*) total_transactions
FROM orders_data
GROUP BY payment_method
ORDER BY total_transactions DESC""")

out = mycursor.fetchall()
print(tabulate(out, headers = [i[0] for i in mycursor.description], tablefmt='psql'))

+------------------+----------------------+
| payment_method   |   total_transactions |
|------------------+----------------------|
| cash             |                 8384 |
| card             |                 8364 |
| upi              |                 8252 |
+------------------+----------------------+


In [25]:
#q6Which date had the highest sales?

mycursor.execute("""SELECT DATE_FORMAT(order_date, '%d-%m-%Y') AS formatted_date, SUM(order_value) AS total_sales
FROM orders_data
GROUP BY order_date
ORDER BY total_sales DESC
LIMIT 10""")

out = mycursor.fetchall()
print(tabulate(out, headers = [i[0] for i in mycursor.description], tablefmt='psql'))

+------------------+-----------------+
| formatted_date   |     total_sales |
|------------------+-----------------|
| 00-00-0000       |     2.43878e+07 |
| 27-10-0177       | 94855.8         |
| 23-07-0175       | 92188.8         |
| 27-02-0176       | 77199.3         |
+------------------+-----------------+
